Building an End-to-End Data Engineering Pipeline for E-Commerce Order Analytics

Step 1: EXTRACT - Baca Data Mentah

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np  
import ast
# Import semua data raw 
data = pd.read_csv("../data/raw/raw_returns.csv")
data.head(37)

,return_id,order_id,alasan_return,status_return,quantity_return,refund_amount,tanggal_return
0,RET-2001,ORD-10070,Produk tidak cocok di kulit,pending,1.0,34000.0,2024-06-07
1,RET-2002,ORD-10057,PRODUK TIDAK COCOK,PENDING,1.0,149000.0,27/07/2024
2,RET-2003,ORD-10045,iritasi ringan,PENDING,3.0,45000.0,"May 13, 2024"
3,RET-2004,ORD-10096,duplikat pesanan,approved,3.0,NaN,2024-05-27
4,RET-2005,ORD-10019,Salah varian,PENDING,2.0,65000.0,2024-07-14
5,RET-2006,ORD-10125,Alergi kandungan produk,rejected,1.0,38000.0,"May 11, 2024"
6,RET-2007,ORD-10101,Berubah pikiran,PENDING,2.0,65000.0,"Jun 06, 2024"
7,RET-2008,ORD-10094,iritasi ringan,approved,3.0,45000.0,2024-05-20
8,RET-2009,ORD-10130,tidak sesuai deskripsi,approved,NaN,NaN,"Jul 11, 2024"
9,RET-2010,ORD-10123,Alergi kandungan produk,rejected,NaN,65000.0,2024-07-14


In [2]:
# Inspeksi awal
print(f"Jumlah baris: {len(data)}")
print(f"Kolom: {list(data.columns)}")
data.info()

Jumlah baris: 37
Kolom: ['return_id', 'order_id', 'alasan_return', 'status_return', 'quantity_return', 'refund_amount', 'tanggal_return']
<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        37 non-null     str    
 1   order_id         37 non-null     str    
 2   alasan_return    35 non-null     str    
 3   status_return    37 non-null     str    
 4   quantity_return  31 non-null     float64
 5   refund_amount    32 non-null     float64
 6   tanggal_return   37 non-null     str    
dtypes: float64(2), str(5)
memory usage: 2.2 KB


In [3]:
print("\nDuplikasi")
print(f"{data.duplicated().sum()}")


Duplikasi
1


In [4]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
return_id          0
order_id           0
alasan_return      2
status_return      0
quantity_return    6
refund_amount      5
tanggal_return     0
dtype: int64


In [5]:
data.nunique()

return_id          36
order_id           36
alasan_return      11
status_return       7
quantity_return     3
refund_amount      10
tanggal_return     35
dtype: int64

In [6]:
print(f"\nHarga negatif: {(data['refund_amount'] < 0).sum()}")


Harga negatif: 3


In [7]:
refund_negatif = data[data['refund_amount'] < 0]

display(refund_negatif)

,return_id,order_id,alasan_return,status_return,quantity_return,refund_amount,tanggal_return
13,RET-2014,ORD-10124,Alergi kandungan produk,pending,3.0,-25000.0,2024-05-28
23,RET-2024,ORD-10087,Alergi kandungan produk,PENDING,1.0,-45000.0,"May 20, 2024"
27,RET-2028,ORD-10074,Berubah pikiran,approved,1.0,-32000.0,2024-06-03


Step 2: TRANSFORM - Bersihkan Data

In [8]:
# Dulikat dan Melihatanya di lokasi berapa?
duplicate = data[data.duplicated()]
display(duplicate)
# Lihat data yang duplikat
print(data[data.duplicated()])

,return_id,order_id,alasan_return,status_return,quantity_return,refund_amount,tanggal_return
35,RET-2006,ORD-10125,Alergi kandungan produk,rejected,1.0,38000.0,"May 11, 2024"


   return_id   order_id            alasan_return status_return  \
35  RET-2006  ORD-10125  Alergi kandungan produk      rejected   

    quantity_return  refund_amount tanggal_return  
35              1.0        38000.0   May 11, 2024  


In [9]:
duplicate_index = data[data.duplicated()].index
print(duplicate_index)

RangeIndex(start=35, stop=36, step=1)


In [10]:
# Mengubah data Negatif menjadi Nilai real atau Positif
data.loc[data['refund_amount'] < 0, 'refund_amount'] = (
    data.loc[data['refund_amount'] < 0, 'refund_amount'].abs())

In [11]:
# Mengubah alasan_return menjadi huruf kecil
data['alasan_return'] = data['alasan_return'].str.lower().str.strip()
# Mengubah status_return menjadi huruf awal kapital
data['status_return'] = data['status_return'].str.strip().str.capitalize()

print(data[['alasan_return', 'status_return']].head(10))

                 alasan_return status_return
0  produk tidak cocok di kulit       Pending
1           produk tidak cocok       Pending
2               iritasi ringan       Pending
3             duplikat pesanan      Approved
4                 salah varian       Pending
5      alergi kandungan produk      Rejected
6              berubah pikiran       Pending
7               iritasi ringan      Approved
8       tidak sesuai deskripsi      Approved
9      alergi kandungan produk      Rejected


In [12]:
# Missing Value akan di handle dengan menghapus data nya saja karena memang tidak 
# lebuh dari 5% dan lebih baik untuk data seperti ini jika di hapus saja
data = data.dropna()
data.head

<bound method NDFrame.head of    return_id   order_id                alasan_return status_return  \
0   RET-2001  ORD-10070  produk tidak cocok di kulit       Pending   
1   RET-2002  ORD-10057           produk tidak cocok       Pending   
2   RET-2003  ORD-10045               iritasi ringan       Pending   
4   RET-2005  ORD-10019                 salah varian       Pending   
5   RET-2006  ORD-10125      alergi kandungan produk      Rejected   
6   RET-2007  ORD-10101              berubah pikiran       Pending   
7   RET-2008  ORD-10094               iritasi ringan      Approved   
10  RET-2011  ORD-10003               iritasi ringan      Rejected   
12  RET-2013  ORD-10114              berubah pikiran     Processed   
13  RET-2014  ORD-10124      alergi kandungan produk       Pending   
14  RET-2015  ORD-10120           produk tidak cocok     Processed   
15  RET-2016  ORD-10053        rusak saat pengiriman      Approved   
16  RET-2017  ORD-10012              berubah pikiran     Pro

In [13]:
# Mengubah angka menjadi bilangan bulat pada kolom quantity_return	dan refund_amount
print(data[['quantity_return', 'refund_amount']].isna().sum())
data['quantity_return'] = data['quantity_return'].astype(int)
data['refund_amount'] = data['refund_amount'].astype(int)
data.head()

quantity_return    0
refund_amount      0
dtype: int64


,return_id,order_id,alasan_return,status_return,quantity_return,refund_amount,tanggal_return
0,RET-2001,ORD-10070,produk tidak cocok di kulit,Pending,1,34000,2024-06-07
1,RET-2002,ORD-10057,produk tidak cocok,Pending,1,149000,27/07/2024
2,RET-2003,ORD-10045,iritasi ringan,Pending,3,45000,"May 13, 2024"
4,RET-2005,ORD-10019,salah varian,Pending,2,65000,2024-07-14
5,RET-2006,ORD-10125,alergi kandungan produk,Rejected,1,38000,"May 11, 2024"


In [14]:
print(data[['tanggal_return']])

   tanggal_return
0      2024-06-07
1      27/07/2024
2    May 13, 2024
4      2024-07-14
5    May 11, 2024
6    Jun 06, 2024
7      2024-05-20
10   Jul 29, 2024
12     2024-05-03
13     2024-05-28
14     11/05/2024
15   Jun 25, 2024
16     30/05/2024
17     2024-05-02
19   Jul 28, 2024
21     2024-07-21
22     21/05/2024
23   May 20, 2024
26     2024-07-01
27     2024-06-03
29   Jun 08, 2024
30     15/06/2024
31     18/07/2024
32   May 26, 2024
34   Jun 27, 2024
35   May 11, 2024
36     2024-06-15


In [15]:
# Normalisasi tanggal return pada kolom tanggal_return menjadi 2024-12-06
# Mengubah berbagai format tanggal menjadi datetime
data['tanggal_return'] = pd.to_datetime(
    data['tanggal_return'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)
data['tanggal_return'] = data['tanggal_return'].dt.strftime('%Y-%m-%d')
print(data['tanggal_return'].isna().sum())

0


In [16]:
print(f"Jumlah baris: {len(data)}")
data.info

Jumlah baris: 27


<bound method DataFrame.info of    return_id   order_id                alasan_return status_return  \
0   RET-2001  ORD-10070  produk tidak cocok di kulit       Pending   
1   RET-2002  ORD-10057           produk tidak cocok       Pending   
2   RET-2003  ORD-10045               iritasi ringan       Pending   
4   RET-2005  ORD-10019                 salah varian       Pending   
5   RET-2006  ORD-10125      alergi kandungan produk      Rejected   
6   RET-2007  ORD-10101              berubah pikiran       Pending   
7   RET-2008  ORD-10094               iritasi ringan      Approved   
10  RET-2011  ORD-10003               iritasi ringan      Rejected   
12  RET-2013  ORD-10114              berubah pikiran     Processed   
13  RET-2014  ORD-10124      alergi kandungan produk       Pending   
14  RET-2015  ORD-10120           produk tidak cocok     Processed   
15  RET-2016  ORD-10053        rusak saat pengiriman      Approved   
16  RET-2017  ORD-10012              berubah pikiran     P

In [17]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
return_id          0
order_id           0
alasan_return      0
status_return      0
quantity_return    0
refund_amount      0
tanggal_return     0
dtype: int64


In [18]:
# Simpan ke CSV data setelah proses atau clean
data.to_csv(
    "../data/warehouse/returns_clean.csv",
    index=False,
    encoding="utf-8"
)
# Melihat kembali data yang disimpan
data = pd.read_csv("../data/warehouse/returns_clean.csv")
data.head()

,return_id,order_id,alasan_return,status_return,quantity_return,refund_amount,tanggal_return
0,RET-2001,ORD-10070,produk tidak cocok di kulit,Pending,1,34000,2024-07-06
1,RET-2002,ORD-10057,produk tidak cocok,Pending,1,149000,2024-07-27
2,RET-2003,ORD-10045,iritasi ringan,Pending,3,45000,2024-05-13
3,RET-2005,ORD-10019,salah varian,Pending,2,65000,2024-07-14
4,RET-2006,ORD-10125,alergi kandungan produk,Rejected,1,38000,2024-05-11
